# Notebook 06 — Paired Delta Analysis Across Phases

Computes paired (per-seed) ΔRMSE and ΔR² between every interesting pair of phases.

Why paired and not unpaired: on small datasets like ESOL, the dominant noise source is the scaffold split itself (per-seed variance ~0.13 RMSE in Phase 1). Same seed → same scaffold split, so the variance of (B − A) for the same seed cancels that noise. The result is a tighter, more interpretable Δ than `mean(B) − mean(A)`.

Reads phase summary JSONs produced by the `save_summary_json` pattern (see `src/reporting.py`). Phases 1 and 2 were backfilled post-hoc with the same schema, see commit `P1 backfill` / `P2 backfill`.

**Outputs:**
- Console: comparison table + per-seed detail
- `reports/paired_deltas.md` — drop-in markdown for the README
- `reports/paired_deltas.json` — structured data for downstream automation

When a new phase summary lands (e.g. `phase5_chemprop`), add its name to `PHASES_TO_LOAD` and the relevant pairs to `PAIRS`. The notebook then regenerates all deltas including those involving the new phase.

## 1. Setup

In [1]:
import os
import sys
import json
from pathlib import Path

# Detect runtime environment
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/qsar-esol-solubility')
else:
    PROJECT_ROOT = Path.cwd().parent

REPORTS_DIR = PROJECT_ROOT / 'reports'

# Make src/ importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.reporting import load_summary_json, paired_delta

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Reports dir : {REPORTS_DIR}")

Mounted at /content/drive
Environment: Colab
Project root: /content/drive/MyDrive/qsar-esol-solubility
Reports dir : /content/drive/MyDrive/qsar-esol-solubility/reports


## 2. Load available phase summaries

Skips phases whose JSON doesn't exist yet (so the notebook still runs partway before Phase 5 is finished).

In [2]:
PHASES_TO_LOAD = ['phase1', 'phase2', 'phase3', 'phase4','phase5']

summaries = {}
for phase in PHASES_TO_LOAD:
    path = REPORTS_DIR / f'{phase}_summary.json'
    if path.exists():
        summaries[phase] = load_summary_json(path)
        agg = summaries[phase]['aggregate']
        n_seeds = len(summaries[phase]['per_seed'])
        print(f"✓ {phase}: {n_seeds} seeds, "
              f"RMSE={agg['RMSE_mean']:.3f}±{agg['RMSE_std']:.3f}, "
              f"R²={agg['R2_mean']:+.3f}±{agg['R2_std']:.3f}")
    else:
        print(f"✗ {phase}: {path} not found — skipping")

print(f"\nLoaded {len(summaries)} summaries: {sorted(summaries.keys())}")

✓ phase1: 5 seeds, RMSE=1.591±0.126, R²=+0.344±0.066
✓ phase2: 5 seeds, RMSE=1.062±0.053, R²=+0.705±0.041
✓ phase3: 5 seeds, RMSE=0.862±0.026, R²=+0.807±0.017
✓ phase4: 5 seeds, RMSE=1.436±0.102, R²=+0.466±0.045
✓ phase5: 5 seeds, RMSE=1.148±0.030, R²=+0.656±0.036

Loaded 5 summaries: ['phase1', 'phase2', 'phase3', 'phase4', 'phase5']


#Sign convention (from `paired_delta` docstring): negative ΔRMSE is improvement, positive ΔR² is improvement. The `wins` counter reflects the per-seed direction (how many of the 5 seeds Phase B beats Phase A in the appropriate direction).

**Pair design:**

*Original 6 comparisons (Phases 1-4):*

- **P1 → P2**: featurization effect, same model (untuned RF). Isolates the value of RDKit descriptors.
- **P2 → P3**: model effect, same features (Morgan + desc). Isolates the value of XGBoost + Optuna over untuned RF.
- **P1 → P3**: cumulative (features + model + tuning). The full Phase 1 → 3 trajectory.
- **P1 → P4**: model class effect on a fixed featurization (Morgan only). Isolates expressivity gain from MLP over RF without the feature confound. Replaces the unpaired Δ originally in the README.
- **P2 → P4** and **P3 → P4**: how much MLP Morgan-only sacrifices vs phases that get the global descriptors. Expected to be positive (MLP loses) — this measures the cost of the feature ablation.

*Phase 5 additions (ChemProp D-MPNN, graph-based, no global descriptors):*

- **P4 → P5**: representation class effect at fixed feature engineering budget. Both phases use no hand-engineered global descriptors — P4 uses Morgan fingerprint (qualitative substructure encoding), P5 uses learned message-passing over the molecular graph. Isolates the value of learned graph representation over fingerprint encoding, controlling for the absence of global descriptors. Expected: negative (P5 wins) — but the magnitude tells us how much "going graph" is worth.
- **P3 → P5**: graph representation vs gold-standard hand-engineered tabular features. P3 uses XGBoost on Morgan + RDKit descriptors (the project's best tabular baseline); P5 uses ChemProp D-MPNN with no hand-engineered features. Expected: positive (P3 wins) on a small dataset like ESOL — confirms that on 1117 molecules, hand-crafted global descriptors carry information the learned graph representation cannot recover. This is the comparison that *tests whether graph alone is sufficient*.
- **P2 → P5**: graph representation vs untuned RF on hand-engineered features. P2 uses untuned RF on Morgan + descriptors; P5 uses tuned ChemProp on the molecular graph. Closes the loop on "graph vs tabular": even when the tabular side is *untuned* (P2), the global descriptors still carry enough signal that graph-only struggles to compete.
- **P1 → P5**: full trajectory baseline vs ChemProp. P1 is the project's starting baseline (untuned RF on Morgan fingerprint). P5 is the strongest pure-learning approach available. The total improvement (or lack of it) from this comparison captures the practical value added by switching to learned graph representations on a dataset of this size.

**Interpretive note for the four P5 comparisons.** The pre-registered hypothesis (project context, Phase 5 entry) was that ChemProp would simultaneously reduce the compression bias at both tails of the logS distribution and resolve the polyol-aromatic hybrid bias of Phase 4. If P5 beats P3 (negative ΔRMSE in `P3 → P5`), the graph representation is empirically validated as competitive with hand-engineered features. If P5 loses to P3 (positive ΔRMSE), this confirms the Phase 4 finding that **compression bias at the extremes is feature-class, not curable by switching from Morgan FP to learned graph features alone**.

**Robustness check.** All ten comparisons are computed with the same per-seed pairing (same scaffold split for both phases on each of the 5 seeds), so the std of the paired Δ is tighter than the std of the unpaired difference of means by a factor that depends on the cross-phase correlation in seed difficulty. Direction-consistency (5/5 wins or 0/5 wins) is a stronger signal than just the mean Δ on small datasets.

Skip pairs where either phase isn't loaded (so the notebook still produces partial output).

In [3]:
PAIRS = [
    ('phase1', 'phase2', 'P1 → P2', 'add RDKit descriptors (same RF)'),
    ('phase2', 'phase3', 'P2 → P3', 'tune XGBoost (same features)'),
    ('phase1', 'phase3', 'P1 → P3', 'tuned XGB + desc vs untuned RF + Morgan'),
    ('phase1', 'phase4', 'P1 → P4', 'tuned MLP vs untuned RF (Morgan-only both)'),
    ('phase2', 'phase4', 'P2 → P4', 'MLP Morgan-only vs RF Morgan+desc'),
    ('phase3', 'phase4', 'P3 → P4', 'MLP Morgan-only vs XGB Morgan+desc'),
    # --- Phase 5 (ChemProp D-MPNN) comparisons ---
    ('phase4', 'phase5', 'P4 → P5', 'ChemProp D-MPNN vs MLP (both no global desc)'),
    ('phase3', 'phase5', 'P3 → P5', 'ChemProp graph vs XGBoost Morgan+desc (best tabular)'),
    ('phase2', 'phase5', 'P2 → P5', 'ChemProp graph vs RF Morgan+desc'),
    ('phase1', 'phase5', 'P1 → P5', 'ChemProp vs untuned RF baseline'),
]

deltas = {}
for a, b, label, note in PAIRS:
    if a not in summaries or b not in summaries:
        print(f"⏭  {label}: missing {a} or {b}")
        continue
    d_rmse = paired_delta(summaries[a]['per_seed'], summaries[b]['per_seed'], 'RMSE')
    d_r2   = paired_delta(summaries[a]['per_seed'], summaries[b]['per_seed'], 'R2')
    deltas[label] = {
        'rmse': d_rmse, 'r2': d_r2,
        'note': note,
        'phase_a': a, 'phase_b': b,
    }
    print(f"\n{label}  ({note})")
    print(f"  ΔRMSE = {d_rmse['mean_delta']:+.3f} ± {d_rmse['std_delta']:.3f}  "
          f"({d_rmse['wins']}/{d_rmse['n_seeds']} seeds improved)")
    print(f"  ΔR²   = {d_r2['mean_delta']:+.3f} ± {d_r2['std_delta']:.3f}  "
          f"({d_r2['wins']}/{d_r2['n_seeds']} seeds improved)")


P1 → P2  (add RDKit descriptors (same RF))
  ΔRMSE = -0.529 ± 0.158  (5/5 seeds improved)
  ΔR²   = +0.361 ± 0.095  (5/5 seeds improved)

P2 → P3  (tune XGBoost (same features))
  ΔRMSE = -0.200 ± 0.036  (5/5 seeds improved)
  ΔR²   = +0.102 ± 0.025  (5/5 seeds improved)

P1 → P3  (tuned XGB + desc vs untuned RF + Morgan)
  ΔRMSE = -0.729 ± 0.125  (5/5 seeds improved)
  ΔR²   = +0.462 ± 0.074  (5/5 seeds improved)

P1 → P4  (tuned MLP vs untuned RF (Morgan-only both))
  ΔRMSE = -0.155 ± 0.055  (5/5 seeds improved)
  ΔR²   = +0.122 ± 0.046  (5/5 seeds improved)

P2 → P4  (MLP Morgan-only vs RF Morgan+desc)
  ΔRMSE = +0.374 ± 0.140  (0/5 seeds improved)
  ΔR²   = -0.239 ± 0.078  (0/5 seeds improved)

P3 → P4  (MLP Morgan-only vs XGB Morgan+desc)
  ΔRMSE = +0.574 ± 0.105  (0/5 seeds improved)
  ΔR²   = -0.341 ± 0.055  (0/5 seeds improved)

P4 → P5  (ChemProp D-MPNN vs MLP (both no global desc))
  ΔRMSE = -0.287 ± 0.098  (5/5 seeds improved)
  ΔR²   = +0.190 ± 0.056  (5/5 seeds improved)


## 4. Summary table

Compact comparison view. Negative ΔRMSE = Phase B wins; positive ΔR² = Phase B wins.

In [4]:
print("=" * 78)
print(f"{'Comparison':<12} {'ΔRMSE':>18} {'ΔR²':>18} {'RMSE wins':>11} {'R² wins':>10}")
print("=" * 78)
for label, d in deltas.items():
    r = d['rmse']; r2 = d['r2']
    print(f"{label:<12} "
          f"{r['mean_delta']:+.3f} ± {r['std_delta']:.3f}    "
          f"{r2['mean_delta']:+.3f} ± {r2['std_delta']:.3f}    "
          f"{r['wins']}/{r['n_seeds']:>5}  {r2['wins']}/{r2['n_seeds']:>5}")
print("=" * 78)

Comparison                ΔRMSE                ΔR²   RMSE wins    R² wins
P1 → P2      -0.529 ± 0.158    +0.361 ± 0.095    5/    5  5/    5
P2 → P3      -0.200 ± 0.036    +0.102 ± 0.025    5/    5  5/    5
P1 → P3      -0.729 ± 0.125    +0.462 ± 0.074    5/    5  5/    5
P1 → P4      -0.155 ± 0.055    +0.122 ± 0.046    5/    5  5/    5
P2 → P4      +0.374 ± 0.140    -0.239 ± 0.078    0/    5  0/    5
P3 → P4      +0.574 ± 0.105    -0.341 ± 0.055    0/    5  0/    5
P4 → P5      -0.287 ± 0.098    +0.190 ± 0.056    5/    5  5/    5
P3 → P5      +0.287 ± 0.042    -0.151 ± 0.030    0/    5  0/    5
P2 → P5      +0.087 ± 0.070    -0.049 ± 0.041    1/    5  1/    5
P1 → P5      -0.442 ± 0.110    +0.312 ± 0.066    5/    5  5/    5


## 5. Per-seed detail (sanity check)

If a pair shows a counterintuitive result (e.g. 1/5 wins despite a favorable mean delta), the per-seed table tells you which seed is the outlier. Often it's the same seed that had a degenerate test set in some other phase.

In [5]:
SEEDS = [42, 0, 1, 2, 3]
print("Per-seed RMSE deltas (negative = Phase B improvement)")
print("-" * 60)
print(f"{'Pair':<12} " + " ".join(f"{s:>9}" for s in SEEDS))
for label, d in deltas.items():
    per_seed = {x['seed']: x['delta'] for x in d['rmse']['per_seed_delta']}
    row = " ".join(f"{per_seed.get(s, float('nan')):>+9.3f}" for s in SEEDS)
    print(f"{label:<12} {row}")

Per-seed RMSE deltas (negative = Phase B improvement)
------------------------------------------------------------
Pair                42         0         1         2         3
P1 → P2         -0.599    -0.276    -0.498    -0.763    -0.509
P2 → P3         -0.184    -0.252    -0.190    -0.148    -0.225
P1 → P3         -0.784    -0.527    -0.688    -0.911    -0.734
P1 → P4         -0.219    -0.102    -0.081    -0.164    -0.208
P2 → P4         +0.381    +0.174    +0.416    +0.599    +0.301
P3 → P4         +0.565    +0.425    +0.607    +0.747    +0.526
P4 → P5         -0.203    -0.184    -0.317    -0.458    -0.274
P3 → P5         +0.362    +0.241    +0.289    +0.289    +0.252
P2 → P5         +0.178    -0.011    +0.099    +0.140    +0.027
P1 → P5         -0.422    -0.286    -0.399    -0.622    -0.481


## 6. Generate markdown block for README

Writes a copy-paste-ready table to `reports/paired_deltas.md`, and also dumps the full structured data to `reports/paired_deltas.json` for downstream automation (e.g. blog post auto-gen, or a future CI check that flags regressions).

In [6]:
md_lines = []
md_lines.append("**Paired delta summary (5-seed scaffold, same split per seed):**\n")
md_lines.append("| Comparison | ΔRMSE | ΔR² | RMSE wins | R² wins | Note |")
md_lines.append("|---|---|---|---|---|---|")
for label, d in deltas.items():
    r = d['rmse']; r2 = d['r2']
    md_lines.append(
        f"| {label} | "
        f"{r['mean_delta']:+.3f} ± {r['std_delta']:.3f} | "
        f"{r2['mean_delta']:+.3f} ± {r2['std_delta']:.3f} | "
        f"{r['wins']}/{r['n_seeds']} | "
        f"{r2['wins']}/{r2['n_seeds']} | "
        f"{d['note']} |"
    )
md_block = "\n".join(md_lines)
print(md_block)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'paired_deltas.md').write_text(md_block + "\n")
print(f"\n→ saved: {REPORTS_DIR / 'paired_deltas.md'}")

deltas_json_path = REPORTS_DIR / 'paired_deltas.json'
deltas_json_path.write_text(json.dumps(
    {label: {
        'phase_a': d['phase_a'],
        'phase_b': d['phase_b'],
        'note': d['note'],
        'rmse': d['rmse'],
        'r2': d['r2'],
    } for label, d in deltas.items()},
    indent=2,
))
print(f"→ saved: {deltas_json_path}")

**Paired delta summary (5-seed scaffold, same split per seed):**

| Comparison | ΔRMSE | ΔR² | RMSE wins | R² wins | Note |
|---|---|---|---|---|---|
| P1 → P2 | -0.529 ± 0.158 | +0.361 ± 0.095 | 5/5 | 5/5 | add RDKit descriptors (same RF) |
| P2 → P3 | -0.200 ± 0.036 | +0.102 ± 0.025 | 5/5 | 5/5 | tune XGBoost (same features) |
| P1 → P3 | -0.729 ± 0.125 | +0.462 ± 0.074 | 5/5 | 5/5 | tuned XGB + desc vs untuned RF + Morgan |
| P1 → P4 | -0.155 ± 0.055 | +0.122 ± 0.046 | 5/5 | 5/5 | tuned MLP vs untuned RF (Morgan-only both) |
| P2 → P4 | +0.374 ± 0.140 | -0.239 ± 0.078 | 0/5 | 0/5 | MLP Morgan-only vs RF Morgan+desc |
| P3 → P4 | +0.574 ± 0.105 | -0.341 ± 0.055 | 0/5 | 0/5 | MLP Morgan-only vs XGB Morgan+desc |
| P4 → P5 | -0.287 ± 0.098 | +0.190 ± 0.056 | 5/5 | 5/5 | ChemProp D-MPNN vs MLP (both no global desc) |
| P3 → P5 | +0.287 ± 0.042 | -0.151 ± 0.030 | 0/5 | 0/5 | ChemProp graph vs XGBoost Morgan+desc (best tabular) |
| P2 → P5 | +0.087 ± 0.070 | -0.049 ± 0.041 | 1/5 | 1/5 | C